# DARC-Sampler for CKD Prediction

**Research objective:** Evaluate whether Density-Adaptive Repulsion-Constrained Sampling (DARC) improves Balanced Accuracy (BA) over class weighting and common resampling baselines.

> Important: No sampler can guarantee a BA above 79.82%. Treat 79.82% as a benchmark to test, not a promised result. This notebook uses a new stratified 80/20 split with `random_state=42`; its holdout result is not directly comparable with a paper result unless the original split and preprocessing protocol are identical.

## Dataset
- Input: `ckd01.csv`
- Target: `Diagnosis` (expected labels: 0 and 1)
- Excluded identifiers: `PatientID`, `DoctorInCharge`
- Sampling and preprocessing are fitted only on training partitions.
- The final holdout set is evaluated once after model selection.

## Methods
1. HistGradientBoosting + balanced sample weights
2. RandomOverSampler + HistGradientBoosting
3. SMOTE + HistGradientBoosting
4. DARC-Sampler + HistGradientBoosting
5. DARC ablations: without density weighting, without clustering, and without majority cleaning

The primary metric is Balanced Accuracy. Also report sensitivity, specificity, F1, ROC-AUC, and confusion matrix.

## 1. Install dependencies

Run this cell once in a fresh Kaggle or Colab runtime.

In [ ]:
%pip -q install -U scikit-learn imbalanced-learn pandas numpy matplotlib seaborn

## 2. Imports and reproducibility

In [ ]:
import os, warnings, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, StratifiedShuffleSplit
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    balanced_accuracy_score, accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report,
    roc_curve
)
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.over_sampling import RandomOverSampler, SMOTE

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("scikit-learn:", __import__("sklearn").__version__)
print("imbalanced-learn:", __import__("imblearn").__version__)

## 3. Load the dataset

If running in Kaggle, set `DATA_PATH` to the dataset path. If running locally or in Colab, upload `ckd01.csv` and use its path.

In [ ]:
DATA_PATH = "/kaggle/input/ckd01.csv"  # Change this if your CSV is stored elsewhere.
if not os.path.exists(DATA_PATH):
    DATA_PATH = "/content/ckd01.csv"    # Common Google Colab upload path.
if not os.path.exists(DATA_PATH):
    DATA_PATH = "ckd01.csv"             # Same folder as this notebook.

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
print("\\nMissing values:", int(df.isna().sum().sum()))
print("\\nTarget counts:")
display(df["Diagnosis"].value_counts(dropna=False).sort_index())

## 4. Data validation and target preparation

In [ ]:
TARGET = "Diagnosis"
DROP_COLUMNS = ["PatientID", "DoctorInCharge"]

required = {TARGET}
missing_required = required - set(df.columns)
if missing_required:
    raise ValueError(f"Missing required target column(s): {missing_required}")

if df[TARGET].isna().any():
    raise ValueError("Diagnosis contains missing values. Resolve these before modelling.")

y = df[TARGET].astype(int)
if set(y.unique()) != {0, 1}:
    raise ValueError(f"Expected binary labels 0 and 1, found: {sorted(y.unique())}")

X = df.drop(columns=[TARGET] + [c for c in DROP_COLUMNS if c in df.columns]).copy()

# Convert any remaining non-numeric columns safely. The supplied dataset is expected
# to contain numeric predictors after excluding DoctorInCharge.
non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    raise ValueError(
        "Non-numeric predictors remain: " + str(non_numeric) +
        ". Encode them inside a fold-specific preprocessing pipeline before running."
    )

X = X.replace([np.inf, -np.inf], np.nan)
feature_names = X.columns.tolist()

print("Predictors:", X.shape[1])
print("Class counts:", y.value_counts().to_dict())
print("Imbalance ratio (majority/minority):",
      round(y.value_counts().max() / y.value_counts().min(), 2))
display(pd.DataFrame({
    "dtype": X.dtypes.astype(str),
    "missing": X.isna().sum(),
    "unique_values": X.nunique()
}))

## 5. Create a locked holdout set

The holdout set is kept out of cross-validation, sampler fitting, and threshold selection. Do not repeatedly tune the model based on holdout results.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
print("Development train:", X_train.shape, y_train.value_counts().to_dict())
print("Locked test:", X_test.shape, y_test.value_counts().to_dict())

## 6. Fold-specific preprocessing

All imputing and scaling are fitted on the current training partition only. Low-cardinality integer-coded predictors are treated as categorical during DARC synthesis: their values are copied from the selected anchor rather than interpolated.

In [ ]:
def get_categorical_indices(X_frame):
    idx = []
    for j, col in enumerate(X_frame.columns):
        s = X_frame[col].dropna()
        if len(s) and pd.api.types.is_integer_dtype(s) and s.nunique() <= 10:
            idx.append(j)
    return idx

CAT_INDICES = get_categorical_indices(X_train)
print("Categorical-like feature count:", len(CAT_INDICES))
print("Categorical-like columns:", [feature_names[i] for i in CAT_INDICES])

def fit_transform_pair(X_fit, X_other):
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    A = imputer.fit_transform(X_fit)
    B = imputer.transform(X_other)
    A = scaler.fit_transform(A)
    B = scaler.transform(B)
    return A.astype(float), B.astype(float), imputer, scaler

## 7. DARC-Sampler implementation

DARC combines three operations:

1. **Density-aware anchor selection:** favour minority observations that are difficult to classify but have local minority support.
2. **Cluster-constrained synthesis:** interpolate only between nearby minority observations in the same minority cluster. Categorical-like values are copied from the anchor.
3. **Capped majority cleaning:** remove only highly overlapping majority observations, with a strict maximum removal fraction.

This is a research proposal. Its novelty and effectiveness must be established through literature review, ablation studies, and independent validation. The code does not establish novelty by itself.

In [ ]:
class DARCSampler(BaseEstimator):
    def __init__(
        self, minority_label=0, sampling_ratio=0.50, k=7, n_clusters=4,
        categorical_features=None, max_remove_fraction=0.02,
        use_density=True, use_clustering=True, use_cleaning=True,
        random_state=42
    ):
        self.minority_label = minority_label
        self.sampling_ratio = sampling_ratio
        self.k = k
        self.n_clusters = n_clusters
        self.categorical_features = categorical_features
        self.max_remove_fraction = max_remove_fraction
        self.use_density = use_density
        self.use_clustering = use_clustering
        self.use_cleaning = use_cleaning
        self.random_state = random_state

    def fit_resample(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        rng = np.random.default_rng(self.random_state)

        classes, counts = np.unique(y, return_counts=True)
        if len(classes) != 2:
            raise ValueError("DARCSampler currently supports binary classification only.")
        if self.minority_label not in classes:
            raise ValueError(f"minority_label={self.minority_label} is not present in y.")

        min_label = self.minority_label
        maj_label = classes[classes != min_label][0]
        Xm = X[y == min_label]
        Xmaj = X[y == maj_label]
        n_min, n_maj = len(Xm), len(Xmaj)

        if n_min < 3:
            raise ValueError("At least 3 minority samples are required.")

        # Number of minority samples required for minority/majority ratio.
        target_min = int(np.ceil(self.sampling_ratio * n_maj))
        n_generate = max(0, target_min - n_min)

        cat_idx = np.array(
            self.categorical_features if self.categorical_features is not None else [],
            dtype=int
        )
        cat_idx = cat_idx[(cat_idx >= 0) & (cat_idx < X.shape[1])]
        cont_idx = np.setdiff1d(np.arange(X.shape[1]), cat_idx)

        # Minority neighbourhood and local difficulty.
        k_eff = min(max(2, self.k), len(X) - 1)
        nn_all = NearestNeighbors(n_neighbors=k_eff + 1).fit(X)
        all_dist, all_ind = nn_all.kneighbors(Xm)
        hardness = np.zeros(n_min)
        for i in range(n_min):
            neigh_labels = y[all_ind[i, 1:]]
            hardness[i] = np.mean(neigh_labels == maj_label)

        # Density is inverse mean distance to nearby minority points.
        k_min = min(max(2, self.k), n_min - 1)
        nn_min = NearestNeighbors(n_neighbors=k_min + 1).fit(Xm)
        min_dist, min_ind = nn_min.kneighbors(Xm)
        mean_dist = np.maximum(min_dist[:, 1:].mean(axis=1), 1e-8)
        density = 1.0 / mean_dist
        density = density / max(density.max(), 1e-8)

        # Cluster the minority class, or use one cluster for the ablation.
        if self.use_clustering and n_min >= self.n_clusters:
            ncl = min(self.n_clusters, n_min)
            labels = KMeans(n_clusters=ncl, random_state=self.random_state,
                            n_init=10).fit_predict(Xm)
        else:
            labels = np.zeros(n_min, dtype=int)

        # Anchor weights: difficult observations with local minority support.
        if self.use_density:
            weights = (0.15 + hardness) * (0.20 + 0.80 * density)
        else:
            weights = 0.15 + hardness
        weights = np.maximum(weights, 1e-8)
        weights = weights / weights.sum()

        synthetic = []
        for _ in range(n_generate):
            anchor = int(rng.choice(n_min, p=weights))
            same_cluster = np.where(labels == labels[anchor])[0]
            candidates = same_cluster[same_cluster != anchor]
            if len(candidates) == 0:
                candidates = np.delete(np.arange(n_min), anchor)

            # Choose a nearby minority neighbour, not a random distant point.
            d = np.linalg.norm(Xm[candidates] - Xm[anchor], axis=1)
            neighbour = int(candidates[np.argmin(d)])
            lam = rng.uniform(0.15, 0.85)

            new_row = Xm[anchor].copy()
            if len(cont_idx):
                new_row[cont_idx] = (
                    Xm[anchor, cont_idx] +
                    lam * (Xm[neighbour, cont_idx] - Xm[anchor, cont_idx])
                )
            if len(cat_idx):
                new_row[cat_idx] = Xm[anchor, cat_idx]
            synthetic.append(new_row)

        X_res = np.vstack([X, np.asarray(synthetic)]) if synthetic else X.copy()
        y_res = np.concatenate([y, np.full(len(synthetic), min_label, dtype=y.dtype)])

        # Conservative majority cleaning. A point is eligible only when its
        # neighbourhood is strongly minority-dominated and minority is closer.
        removed = 0
        if self.use_cleaning and len(Xmaj) and self.max_remove_fraction > 0:
            nn_m = NearestNeighbors(n_neighbors=min(k_eff, len(Xm))).fit(Xm)
            nn_M = NearestNeighbors(n_neighbors=min(k_eff, len(Xmaj))).fit(Xmaj)
            dm, im = nn_m.kneighbors(Xmaj)
            dM, _ = nn_M.kneighbors(Xmaj)
            frac_min = np.mean(dm < np.inf, axis=1)  # shape guard
            # Compare local distance scales and use the fraction of minority
            # neighbours among a joint neighbourhood.
            for i in range(len(Xmaj)):
                dist_min = dm[i].min()
                dist_maj = dM[i].min()
                if dist_min < 0.75 * max(dist_maj, 1e-8):
                    # Additional local class-overlap test.
                    joint_d = np.concatenate([dm[i], dM[i]])
                    joint_lab = np.concatenate([
                        np.full(len(dm[i]), min_label),
                        np.full(len(dM[i]), maj_label)
                    ])
                    order = np.argsort(joint_d)[:min(7, len(joint_d))]
                    if np.mean(joint_lab[order] == min_label) >= 0.85:
                        # Mark by original row index in X.
                        maj_positions = np.where(y == maj_label)[0]
                        # Current i corresponds to i-th majority row.
                        eligible_global = maj_positions[i]
                        # Store eligible original row positions.
                        if 'remove_positions' not in locals():
                            remove_positions = []
                        remove_positions.append(eligible_global)

            if 'remove_positions' in locals() and remove_positions:
                cap = int(np.floor(self.max_remove_fraction * n_maj))
                chosen = np.array(sorted(set(remove_positions))[:cap], dtype=int)
                if len(chosen):
                    keep = np.ones(len(X_res), dtype=bool)
                    keep[chosen] = False
                    X_res, y_res = X_res[keep], y_res[keep]
                    removed = len(chosen)
                del remove_positions

        self.statistics_ = {
            "minority_before": int(n_min),
            "majority_before": int(n_maj),
            "synthetic_added": int(n_generate),
            "majority_removed": int(removed),
            "minority_after": int(np.sum(y_res == min_label)),
            "majority_after": int(np.sum(y_res == maj_label)),
        }
        return X_res, y_res

## 8. Model and metric utilities

Thresholds are selected on an internal calibration subset of the outer training fold. The outer validation fold is never used to choose a threshold.

In [ ]:
def make_model():
    return HistGradientBoostingClassifier(
        learning_rate=0.06,
        max_iter=250,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=SEED
    )

def choose_threshold(y_true, probabilities):
    thresholds = np.arange(0.05, 0.951, 0.01)
    scores = [balanced_accuracy_score(y_true, (probabilities >= t).astype(int))
              for t in thresholds]
    best_score = np.max(scores)
    # Prefer a threshold nearest 0.50 when multiple thresholds tie.
    best = [t for t, s in zip(thresholds, scores) if np.isclose(s, best_score)]
    return float(min(best, key=lambda t: abs(t - 0.50)))

def calculate_metrics(y_true, probabilities, threshold):
    pred = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "BA": balanced_accuracy_score(y_true, pred),
        "Accuracy": accuracy_score(y_true, pred),
        "Sensitivity_class1": recall_score(y_true, pred, pos_label=1, zero_division=0),
        "Specificity_class0": recall_score(y_true, pred, pos_label=0, zero_division=0),
        "Precision_class1": precision_score(y_true, pred, pos_label=1, zero_division=0),
        "F1_class1": f1_score(y_true, pred, pos_label=1, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, probabilities),
        "Threshold": threshold,
        "TN": tn, "FP": fp, "FN": fn, "TP": tp
    }

def fit_method(X_fit_raw, y_fit, X_eval_raw, method, seed=42):
    X_fit, X_eval, imputer, scaler = fit_transform_pair(X_fit_raw, X_eval_raw)
    y_fit_arr = np.asarray(y_fit)

    if method == "Weighting":
        X_res, y_res = X_fit, y_fit_arr
        sample_weight = compute_sample_weight(class_weight="balanced", y=y_res)
    else:
        sample_weight = None
        if method == "ROS":
            sampler = RandomOverSampler(random_state=seed)
        elif method == "SMOTE":
            sampler = SMOTE(random_state=seed, k_neighbors=5)
        elif method == "DARC":
            sampler = DARCSampler(
                minority_label=0, sampling_ratio=0.50, k=7, n_clusters=4,
                categorical_features=CAT_INDICES, max_remove_fraction=0.02,
                random_state=seed
            )
        elif method == "DARC_no_density":
            sampler = DARCSampler(
                minority_label=0, sampling_ratio=0.50, k=7, n_clusters=4,
                categorical_features=CAT_INDICES, max_remove_fraction=0.02,
                use_density=False, random_state=seed
            )
        elif method == "DARC_no_clustering":
            sampler = DARCSampler(
                minority_label=0, sampling_ratio=0.50, k=7, n_clusters=4,
                categorical_features=CAT_INDICES, max_remove_fraction=0.02,
                use_clustering=False, random_state=seed
            )
        elif method == "DARC_no_cleaning":
            sampler = DARCSampler(
                minority_label=0, sampling_ratio=0.50, k=7, n_clusters=4,
                categorical_features=CAT_INDICES, max_remove_fraction=0.0,
                use_cleaning=False, random_state=seed
            )
        else:
            raise ValueError("Unknown method: " + method)
        X_res, y_res = sampler.fit_resample(X_fit, y_fit_arr)

    model = make_model()
    if sample_weight is None:
        model.fit(X_res, y_res)
    else:
        model.fit(X_res, y_res, sample_weight=sample_weight)
    return model.predict_proba(X_eval)[:, 1], model, (imputer, scaler)

## 9. Repeated stratified cross-validation

For each outer fold:
- Split the outer training data into an inner fit subset and a calibration subset.
- Fit preprocessing, sampling, and the classifier on the inner fit subset only.
- Select the decision threshold using the untouched calibration subset.
- Refit preprocessing, sampling, and the classifier on the complete outer training fold.
- Evaluate once on the outer validation fold.

Start with 3 repeats to check runtime. Use 5 repeats for the main experiment.

In [ ]:
METHODS = [
    "Weighting", "ROS", "SMOTE", "DARC",
    "DARC_no_density", "DARC_no_clustering", "DARC_no_cleaning"
]
N_SPLITS = 5
N_REPEATS = 3  # Set to 5 for the main reported experiment.

cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED
)
cv_rows = []

for fold_id, (tr_idx, val_idx) in enumerate(cv.split(X_train, y_train), start=1):
    X_outer_train = X_train.iloc[tr_idx]
    y_outer_train = y_train.iloc[tr_idx]
    X_outer_val = X_train.iloc[val_idx]
    y_outer_val = y_train.iloc[val_idx]

    inner_split = StratifiedShuffleSplit(n_splits=1, test_size=0.25,
                                         random_state=SEED + fold_id)
    fit_idx, cal_idx = next(inner_split.split(X_outer_train, y_outer_train))
    X_inner_fit = X_outer_train.iloc[fit_idx]
    y_inner_fit = y_outer_train.iloc[fit_idx]
    X_cal = X_outer_train.iloc[cal_idx]
    y_cal = y_outer_train.iloc[cal_idx]

    for method in METHODS:
        # Inner calibration: threshold selection only.
        p_cal, _, _ = fit_method(
            X_inner_fit, y_inner_fit, X_cal, method, seed=SEED + fold_id
        )
        threshold = choose_threshold(y_cal, p_cal)

        # Refit on the complete outer training fold, then score outer validation.
        p_val, _, _ = fit_method(
            X_outer_train, y_outer_train, X_outer_val, method,
            seed=SEED + fold_id
        )
        metrics = calculate_metrics(y_outer_val, p_val, threshold)
        metrics.update({"Fold": fold_id, "Method": method})
        cv_rows.append(metrics)

    if fold_id % 5 == 0:
        print(f"Completed fold {fold_id}/{N_SPLITS * N_REPEATS}")

cv_results = pd.DataFrame(cv_rows)
cv_results.to_csv("darc_cv_fold_results.csv", index=False)
display(cv_results.head())

## 10. Cross-validation summary

In [ ]:
summary = (cv_results.groupby("Method")
           .agg(
               BA_mean=("BA", "mean"),
               BA_std=("BA", "std"),
               BA_median=("BA", "median"),
               Accuracy_mean=("Accuracy", "mean"),
               Sensitivity_mean=("Sensitivity_class1", "mean"),
               Specificity_mean=("Specificity_class0", "mean"),
               F1_mean=("F1_class1", "mean"),
               AUC_mean=("ROC_AUC", "mean")
           )
           .sort_values("BA_mean", ascending=False))
display(summary.round(4))
summary.to_csv("darc_cv_summary.csv")

plt.figure(figsize=(11, 5))
sns.boxplot(data=cv_results, x="Method", y="BA")
plt.xticks(rotation=35, ha="right")
plt.ylabel("Balanced Accuracy")
plt.title("Repeated CV Balanced Accuracy")
plt.tight_layout()
plt.show()

## 11. Paired comparison against class weighting

Use fold-matched BA differences. A paired t-test is included as a simple exploratory test, but repeated CV folds are correlated, so its p-value can be optimistic. Report the mean paired difference and confidence interval, and consider a corrected resampled test or an independent external cohort for confirmatory claims.

In [ ]:
from scipy.stats import ttest_rel, t

def paired_comparison(results, candidate, reference="Weighting"):
    a = (results[results.Method == candidate]
         .sort_values("Fold")["BA"].to_numpy())
    b = (results[results.Method == reference]
         .sort_values("Fold")["BA"].to_numpy())
    if len(a) != len(b):
        raise ValueError("Fold counts do not match.")
    diff = a - b
    n = len(diff)
    mean_diff = diff.mean()
    se = diff.std(ddof=1) / np.sqrt(n)
    critical = t.ppf(0.975, df=n-1)
    p_value = ttest_rel(a, b).pvalue
    return {
        "Candidate": candidate,
        "Reference": reference,
        "Mean_BA_difference": mean_diff,
        "CI95_low": mean_diff - critical * se,
        "CI95_high": mean_diff + critical * se,
        "Paired_t_test_p": p_value,
        "Candidate_wins": int(np.sum(diff > 0)),
        "Folds": n
    }

comparison = pd.DataFrame([
    paired_comparison(cv_results, m)
    for m in METHODS if m != "Weighting"
])
display(comparison.sort_values("Mean_BA_difference", ascending=False).round(4))
comparison.to_csv("darc_paired_comparison.csv", index=False)

## 12. Select one method using development CV only

The method with the highest mean CV BA is selected. This selection rule must be stated before looking at the locked test results. If two methods are close, prefer the simpler method unless the difference is supported by uncertainty analysis.

In [ ]:
best_method = summary.index[0]
print("Selected method from development CV:", best_method)
print("Development CV mean BA:", round(summary.loc[best_method, "BA_mean"], 4))
print("Benchmark target BA: 0.7982")

## 13. Final threshold selection and one-time holdout evaluation

The threshold is selected using a calibration split from the development training set. The chosen method is then refitted on all development training rows and evaluated once on the locked holdout set.

In [ ]:
final_split = StratifiedShuffleSplit(n_splits=1, test_size=0.25,
                                     random_state=SEED + 999)
fit_idx, cal_idx = next(final_split.split(X_train, y_train))
X_fit = X_train.iloc[fit_idx]
y_fit = y_train.iloc[fit_idx]
X_cal = X_train.iloc[cal_idx]
y_cal = y_train.iloc[cal_idx]

p_cal, _, _ = fit_method(X_fit, y_fit, X_cal, best_method, seed=SEED + 999)
final_threshold = choose_threshold(y_cal, p_cal)
print("Selected threshold:", final_threshold)

p_test, final_model, preprocessing = fit_method(
    X_train, y_train, X_test, best_method, seed=SEED + 1000
)
test_metrics = calculate_metrics(y_test, p_test, final_threshold)
display(pd.DataFrame([test_metrics]).round(4))

test_pred = (p_test >= final_threshold).astype(int)
print(classification_report(y_test, test_pred, digits=4, zero_division=0))
print("Confusion matrix [ [TN, FP], [FN, TP] ]:")
print(confusion_matrix(y_test, test_pred, labels=[0, 1]))

test_result = pd.DataFrame([{"Method": best_method, **test_metrics}])
test_result.to_csv("darc_final_holdout_result.csv", index=False)

plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, test_pred, labels=[0, 1]),
            annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred 0", "Pred 1"],
            yticklabels=["True 0", "True 1"])
plt.title("Locked Holdout Confusion Matrix")
plt.tight_layout()
plt.show()

## 14. Test-set uncertainty

The holdout set is small, especially for the minority class. Report the number of class-0 and class-1 observations beside BA. One additional correctly classified observation in a class containing 27 cases changes that class recall by about 3.70 percentage points and BA by about 1.85 percentage points.

In [ ]:
class_counts_test = y_test.value_counts().sort_index()
print("Holdout class counts:", class_counts_test.to_dict())
print("Holdout BA:", round(test_metrics["BA"], 4))
print("Reference benchmark:", 0.7982)
print("Difference vs benchmark:", round(test_metrics["BA"] - 0.7982, 4))

## 15. Sensitivity analysis for DARC sampling ratio

Run this after the main experiment. It compares minority-to-majority target ratios of 0.33, 0.50, and 0.67 using the same fold-safe threshold protocol. This section is intentionally a smaller experiment; use it to choose a ratio before a confirmatory run.

In [ ]:
def fit_darc_ratio(X_fit_raw, y_fit, X_eval_raw, ratio, seed=42):
    X_fit, X_eval, _, _ = fit_transform_pair(X_fit_raw, X_eval_raw)
    sampler = DARCSampler(
        minority_label=0, sampling_ratio=ratio, k=7, n_clusters=4,
        categorical_features=CAT_INDICES, max_remove_fraction=0.02,
        random_state=seed
    )
    X_res, y_res = sampler.fit_resample(X_fit, np.asarray(y_fit))
    model = make_model().fit(X_res, y_res)
    return model.predict_proba(X_eval)[:, 1]

ratio_rows = []
ratio_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=1, random_state=SEED)
for fold_id, (tr, va) in enumerate(ratio_cv.split(X_train, y_train), start=1):
    Xtr, ytr = X_train.iloc[tr], y_train.iloc[tr]
    Xva, yva = X_train.iloc[va], y_train.iloc[va]
    inner = StratifiedShuffleSplit(n_splits=1, test_size=0.25,
                                   random_state=SEED + 2000 + fold_id)
    fi, ci = next(inner.split(Xtr, ytr))
    for ratio in [0.33, 0.50, 0.67]:
        pcal = fit_darc_ratio(Xtr.iloc[fi], ytr.iloc[fi],
                              Xtr.iloc[ci], ratio, SEED + fold_id)
        th = choose_threshold(ytr.iloc[ci], pcal)
        pva = fit_darc_ratio(Xtr, ytr, Xva, ratio, SEED + fold_id)
        row = calculate_metrics(yva, pva, th)
        row.update({"Fold": fold_id, "Ratio": ratio})
        ratio_rows.append(row)

ratio_results = pd.DataFrame(ratio_rows)
display(ratio_results.groupby("Ratio")["BA"].agg(["mean", "std"]).round(4))
ratio_results.to_csv("darc_ratio_sensitivity.csv", index=False)

## 16. Research reporting checklist

Before claiming that DARC beats the 79.82% benchmark:

- [ ] Confirm that the target coding is correct: `0 = Non-CKD`, `1 = CKD`.
- [ ] Confirm the original paper's exact train/test split and reproduce its baseline under the same protocol.
- [ ] Use at least 5 repeated stratified CV repeats for the main comparison.
- [ ] Keep preprocessing, sampler fitting, and threshold selection inside training partitions.
- [ ] Report mean BA, standard deviation, paired fold differences, and uncertainty intervals.
- [ ] Report sensitivity and specificity separately. A high BA can hide poor performance for one class.
- [ ] Run ablations and sampling-ratio sensitivity analysis.
- [ ] Evaluate the locked test set once.
- [ ] Validate on an independent cohort before making strong clinical generalization claims.
- [ ] Review the literature to establish whether the DARC mechanism is genuinely novel.

**Do not claim statistical significance from the paired t-test alone.** Repeated cross-validation folds are dependent. A confirmatory study should use an independent test cohort or a statistically appropriate corrected procedure.

## 17. Export results

The notebook writes these files in the current working directory:
- `darc_cv_fold_results.csv`
- `darc_cv_summary.csv`
- `darc_paired_comparison.csv`
- `darc_final_holdout_result.csv`
- `darc_ratio_sensitivity.csv`

You can download them from the notebook output panel or package them into a ZIP.

In [ ]:
import zipfile
result_files = [
    "darc_cv_fold_results.csv",
    "darc_cv_summary.csv",
    "darc_paired_comparison.csv",
    "darc_final_holdout_result.csv",
    "darc_ratio_sensitivity.csv"
]
existing = [f for f in result_files if os.path.exists(f)]
if existing:
    with zipfile.ZipFile("darc_experiment_results.zip", "w",
                         compression=zipfile.ZIP_DEFLATED) as zf:
        for f in existing:
            zf.write(f)
    print("Created darc_experiment_results.zip with:", existing)
else:
    print("Run the experiment cells first to generate result files.")